# ⚡ Host Aegis for FREE from Colab (no card, real model)

This runs your full trading app (torch model + dashboard) on Colab's free RAM and
exposes it on a public URL via a free Cloudflare tunnel — **no credit card, no signup.**

**How:** just run the two cells below. The second cell prints a link like
`https://something.trycloudflare.com` — open it and add `/dashboard/`.

> The app stays live **as long as this Colab tab is running.** Colab disconnects when
> idle (~90 min) or after ~12 h — fine for testing/demoing. For 24/7 you need a paid host.
> Not financial advice; still the ~51.6% model.

## 1. Get the code + deps

In [ ]:
import os
if not os.path.exists('Ai_pred'):
    !git clone -q https://github.com/SuriyaDcruze/Ai_pred.git
%cd Ai_pred
# Colab already has torch/numpy/pandas/scikit-learn — add only the light deps.
!pip install -q fastapi 'uvicorn[standard]' pydantic pydantic-settings httpx websockets rich python-dotenv
print('ready')

## 2. Start the server + public tunnel

Watch the output for the **`https://….trycloudflare.com`** link, then open it and add **`/dashboard/`**.

In [ ]:
import subprocess, time, os, urllib.request
os.environ['AEGIS_DEVICE'] = 'cpu'

# 1) start the app in the background on port 8000
server = subprocess.Popen(['uvicorn', 'app.api.main:app', '--host', '0.0.0.0', '--port', '8000'])

# 2) wait until it answers /health
for _ in range(60):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2).status == 200:
            print('✅ server up'); break
    except Exception:
        time.sleep(2)

# 3) download cloudflared and open a free public tunnel (no login)
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print('\n🌐 Your public link is below (open it + add /dashboard/):\n')
!./cloudflared tunnel --url http://localhost:8000 2>&1 | grep -i --line-buffered 'trycloudflare.com\|error'